# First Model - Ridge Regression

**Why this model?** We chose Ridge Regression as it is a useful linear baseline for our task. The EDA revealed strong relationships between calendar features, a distinct interaction between hour and day type, and a non-linear relationship between temperature and demand. We therefore need to encode these patterns explicitly for a linear model. In addition, several calendar features are structurally dependent, and one-hot encoding introduces correlated predictors. Ridge addresses this by adding an L2 penalty, which helps stabilise coefficient estimates and reduces sensitivity to multicollinearity. Numerical features must also be scaled because the regularisation penalty depends on their magnitude. This model will help us assess how well a regularised linear approach captures demand patterns and how its predictive performance differs between random and chronological validation.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (FunctionTransformer, OneHotEncoder, PolynomialFeatures, StandardScaler)

from src.data import load_data
from src.validation import RESULTS_DIR, SCHEMES, evaluate, get_cv

In [2]:
X, y = load_data()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

display(X.head())
display(y.head())

Features shape: (17379, 9)
Target shape: (17379,)


,mnth,hr,holiday,weekday,workingday,weathersit,temp,hum,windspeed
datetime,,,,,,,,,
2011-01-01 00:00:00,1,0,0,6,0,1,0.24,0.81,0.0
2011-01-01 01:00:00,1,1,0,6,0,1,0.22,0.80,0.0
2011-01-01 02:00:00,1,2,0,6,0,1,0.22,0.80,0.0
2011-01-01 03:00:00,1,3,0,6,0,1,0.24,0.75,0.0
2011-01-01 04:00:00,1,4,0,6,0,1,0.24,0.75,0.0


datetime
2011-01-01 00:00:00    16
2011-01-01 01:00:00    40
2011-01-01 02:00:00    32
2011-01-01 03:00:00    13
2011-01-01 04:00:00     1
Name: cnt, dtype: int64

In [3]:
print("Chronological order:", X.index.is_monotonic_increasing)
print("Unique timestamps:", X.index.is_unique)
print("'yr' excluded:", "yr" not in X.columns)
print("Target aligned with features:", X.index.equals(y.index))

print("\nRemaining missing values:")
print(X.isna().sum()[X.isna().sum() > 0])

Chronological order: True
Unique timestamps: True
'yr' excluded: True
Target aligned with features: True

Remaining missing values:
hum    22
dtype: int64


In [4]:
def make_hour_day_category(X):
    X = np.asarray(X)

    hour = X[:, 0].astype(int).astype(str)
    workingday = X[:, 1].astype(int).astype(str)

    return (hour + "_" + workingday).reshape(-1, 1)

In [5]:
# Categorical features
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# Hour × workingday interaction
hour_day_transformer = Pipeline([
    ("combine", FunctionTransformer(make_hour_day_category, validate=False)),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Numerical features with missing values
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Temperature with a quadratic term
temperature_transformer = Pipeline([
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler())
])

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_transformer, ["mnth", "weekday", "weathersit"]),
        ("hour_day", hour_day_transformer, ["hr", "workingday"]),
        ("binary", "passthrough", ["holiday", "workingday"]),
        ("numeric", numeric_transformer, ["hum", "windspeed"]),
        ("temperature", temperature_transformer, ["temp"])
    ],
    remainder="drop"
)

In [7]:
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge())
])

In [8]:
ALPHAS = np.logspace(-5, 3, 25)
INNER_SPLITS = 3


def make_ridge_search(scheme, log_target=False):
    """Ridge pipeline tuned with an inner CV of the same type as the outer scheme."""
    if log_target:
        estimator = TransformedTargetRegressor(
            regressor=ridge_pipeline, func=np.log1p, inverse_func=np.expm1
        )
        param_grid = {"regressor__model__alpha": ALPHAS}
    else:
        estimator = ridge_pipeline
        param_grid = {"model__alpha": ALPHAS}

    return GridSearchCV(
        estimator,
        param_grid,
        cv=get_cv(scheme, n_splits=INNER_SPLITS),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )

In [9]:
variants = {"ridge": False, "ridge_log": True}

results = {}
for name, log_target in variants.items():
    for scheme in SCHEMES:
        model = make_ridge_search(scheme, log_target)
        results[(name, scheme)] = evaluate(model, X, y, scheme, name)

In [10]:
def summarise(name, scheme):
    folds = pd.read_csv(RESULTS_DIR / f"{name}_{scheme}_folds.csv")
    return {
        "model": name,
        "scheme": scheme,
        "rmse_mean": folds["rmse"].mean(),
        "rmse_std": folds["rmse"].std(),
        "mae_mean": folds["mae"].mean(),
        "r2_mean": folds["r2"].mean(),
    }


names = ["baseline_hour_day", "ridge", "ridge_log"]
summary = pd.DataFrame([summarise(n, s) for n in names for s in SCHEMES])
summary.round(3)

,model,scheme,rmse_mean,rmse_std,mae_mean,r2_mean
0,baseline_hour_day,random,109.942,1.384,72.959,0.632
1,baseline_hour_day,chrono,125.070,36.325,88.837,0.502
2,ridge,random,89.096,1.256,65.317,0.758
3,ridge,chrono,110.747,32.262,84.136,0.624
4,ridge_log,random,81.478,0.926,52.267,0.798
5,ridge_log,chrono,109.259,35.150,75.553,0.629


In [11]:
for (name, scheme), (folds, _) in results.items():
    print(name, scheme)
    print(folds[["fold", "best_params"]].to_string(index=False), "\n")

ridge random
 fold                                        best_params
    1                  {'model__alpha': np.float64(1.0)}
    2  {'model__alpha': np.float64(0.46415888336127725)}
    3 {'model__alpha': np.float64(0.004641588833612777)}
    4  {'model__alpha': np.float64(0.21544346900318823)}
    5  {'model__alpha': np.float64(0.46415888336127725)} 

ridge chrono
 fold                                          best_params
    1  {'model__alpha': np.float64(4.641588833612782e-05)}
    2  {'model__alpha': np.float64(4.641588833612782e-05)}
    3                 {'model__alpha': np.float64(0.0001)}
    4 {'model__alpha': np.float64(0.00021544346900318823)}
    5                  {'model__alpha': np.float64(1e-05)} 

ridge_log random
 fold                                                     best_params
    1 {'regressor__model__alpha': np.float64(2.1544346900318823e-05)}
    2 {'regressor__model__alpha': np.float64(0.00046415888336127773)}
    3  {'regressor__model__alpha': np.float64(4

In [12]:
for (name, scheme), (_, preds) in results.items():
    share = (preds["y_pred"] < 0).mean()
    print(f"{name:10s} {scheme:7s} negative predictions: {share:.2%}")

ridge      random  negative predictions: 9.94%
ridge      chrono  negative predictions: 9.70%
ridge_log  random  negative predictions: 0.00%
ridge_log  chrono  negative predictions: 0.00%


In [13]:
results[("ridge", "chrono")][0].drop(columns="best_params")

,fold,n_train,n_test,test_start,test_end,rmse,mae,r2,rmsle
0,1,2899,2896,2011-05-05 17:00:00,2011-09-03 21:00:00,91.325007,75.150038,0.629837,1.073084
1,2,5795,2896,2011-09-03 22:00:00,2012-01-02 22:00:00,63.772318,46.891932,0.768307,0.928989
2,3,8691,2896,2012-01-02 23:00:00,2012-05-02 23:00:00,124.840050,96.019769,0.506214,1.511228
3,4,11587,2896,2012-05-03 00:00:00,2012-08-31 15:00:00,139.469270,101.777023,0.597553,0.741882
4,5,14483,2896,2012-08-31 16:00:00,2012-12-31 23:00:00,134.327276,100.841028,0.619194,1.181906
